# 第7回 教師あり解析のパイプラインを実装する（解答例・教員用）


---


## 前回からの接続


前回は、心臓病かどうかを確率で当て、しきい値0.5で判定しました。61人のテストで、見逃しは2人でした。

前回の最後に、言えないことが2つ残りました。しきい値はどこがよいかと、数値の顔をした区分（胸の痛みの型`cp`など）をどう扱うか、です。今日はこの2つに答えます。そして、前処理とモデルを1本の部品にまとめ、新しい受診者を同じ手順で判定できるようにします。


---


## 今日の分析目標


あなたは健診センターの医師です。新しい受診者を、訓練と同じ手順で判定したい。そして、どの確率から精密検査に回すかを、誤りの重さから決めたい。

前処理とモデルを1本にまとめ、しきい値を動かしたときの見逃しと誤検出を表にし、誤りの重さから区切りを決めます。もう1つのねらいは、`fit`や`predict`の正体（クラスとメソッド）を読めるようになることです。最後の「目標に答えられたか」で振り返りましょう。


### 場面

あなたは、今日も健診センターの医師です。今日は、新しい受診者が来ます。心臓カテーテルの精密検査は体への負担が大きいので、気軽には回せません。

決めたいのは2つです。新しい人にも、訓練と同じ手順で判定すること。そして、どの確率から精密検査に回すかを、誤りの重さから決めることです。病気の人を健康だとして帰す見逃しは、治療の機会を失わせます。健康な人を病気と言う誤検出は、再検査で済みます。


### 手元のデータ

前回と同じ、米国クリーブランドの病院で心臓の検査を受けた303人の記録です（UCI Machine Learning RepositoryのHeart Disease）。1行が1人で、年齢・性別・胸の痛みの型など、13の検査値が並びます。

答えの列`target`は、心臓カテーテルの検査で、太い血管の少なくとも1本が半分以上狭まっていた人を1（心臓病あり）、そうでない人を0としています。


### 最後に出す答え

「確率がいくつを超えたら精密検査に回すか」という区切りと、新しい受診者の確率と、その区切りでの判定です。


---


## 学習ゴール


この回を終えると、次のことができるようになります。

- 番号で書かれた区分を、ワンホット（one-hot）で列に分けられる
- 前処理とモデルをパイプライン（pipeline）にまとめ、新しいデータに同じ手順を当てられる
- `fit`と`transform`を持つ小さなクラスを読み、scikit-learnの部品と同じ形だと説明できる
- GridSearchCVで、交差検証（cross-validation）をしながらCを選べる
- しきい値を動かしたときの見逃しと誤検出を表にし、誤りの重さからしきい値を決められる


---


## 0. 設定


### 最初に実行する：道具を呼び出す設定のセル

- 役割：このノートで使う道具を呼び出す
- 入力：なし
- 出力：「セットアップ完了」の表示


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import roc_auc_score

!pip install -q japanize-matplotlib   # 図中の日本語が □ になるのを防ぐ
try:
    import japanize_matplotlib
except Exception:   # japanize が動かなくなったときの保険：同梱フォントを直接登録する
    from importlib.util import find_spec
    from matplotlib import font_manager as fm
    from pathlib import Path
    spec = find_spec('japanize_matplotlib')
    ttf = next(Path(spec.origin).parent.rglob('*.ttf'), None) if spec else None
    if ttf:
        fm.fontManager.addfont(str(ttf))
        plt.rcParams['font.family'] = fm.FontProperties(fname=str(ttf)).get_name()

plt.rcParams['figure.figsize'] = (7, 5)
plt.rcParams['font.size'] = 13
plt.rcParams['axes.unicode_minus'] = False   # マイナス記号の化けを防ぐ
print('セットアップ完了')


### 分析の地図：今日はここ

データ解析は「① データの理解と目標の設定 → ② 前処理（preprocessing）とデータ解析 → ③ 結果の解釈と目標との整合」の3つのフェーズを回ります。今日は ② の前処理と解析をひとまとめにし、③ で区切りを決めます。


In [ ]:
# @title 図：分析の地図
fig, ax = plt.subplots(figsize=(10, 2.6))
ax.axis('off')
phases = ['① データの理解と\n目標の設定', '② 前処理と\nデータ解析', '③ 結果の解釈と\n目標との整合']
colors = ['#0066cc', '#2a9d8f', '#e63946']
for i, (x0, p, c) in enumerate(zip([0.17, 0.5, 0.83], phases, colors), start=1):
    on = i in (2, 3)
    ax.text(x0, 0.62, p, ha='center', va='center', fontsize=13 if on else 11, color='white',
            bbox=dict(boxstyle='round,pad=0.6', facecolor=c, alpha=0.95 if on else 0.3))
for a0, a1 in [(0.29, 0.365), (0.62, 0.695)]:
    ax.annotate('', xy=(a1, 0.62), xytext=(a0, 0.62),
                arrowprops=dict(arrowstyle='->', color='#555', lw=2))
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
plt.show()


---


## 1. 前処理とモデルを、1本にまとめる


前回は`prep`で整え、`cp`などの区分は番号のまま入れていたので、今日は区分の扱いを決めて、前処理とモデルを1本にまとめます。データは前回と同じ、303人の心臓の検査です。今日は、列を数の列と区分の列に分けて扱います。


### お手本：読み込んで、訓練とテストに分ける

- 役割：データを読み込み、数の列と区分の列を分け、訓練とテストに分ける
- 入力：データの置き場所とファイル名
- 出力：`X_tr`・`X_te`（表のまま）と、答え`y_tr`・`y_te`


In [ ]:
# データはこのリポジトリから読み込む
DATA_DIR = ('https://raw.githubusercontent.com/k0heiun0/'
            'applied_exercise_preview/main/shared/data')
df = pd.read_csv(f'{DATA_DIR}/heart.csv')
num = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']       # 数の列
cat = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']
X, y = df[num + cat], df['target'].values
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2,
                                          random_state=42, stratify=y)
print(len(y_tr), len(y_te))


訓練242人、テスト61人。前回と同じ分け方です。

胸の痛みの型`cp`は1〜4の番号ですが、これは区分の名前です。1と2の差と3と4の差が同じという意味はありません。番号のまま入れると、係数は「番号が1増えるごとに」の傾きになってしまいます。


### お手本：区分を、ワンホットで列に分ける

- 役割：`cp`の4つの区分を、「その区分なら1、違えば0」の4本の列に分ける
- 入力：訓練の`cp`列
- 出力：先頭3人の、分けた4本の列


In [ ]:
oh = OneHotEncoder().fit(X_tr[['cp']])
print(X_tr['cp'].head(3).tolist())
print(oh.transform(X_tr[['cp']].head(3)).toarray())


先頭3人の`cp`は4, 2, 2。1人目は4本目の列だけが1、2人目と3人目は2本目の列だけが1です。これで、区分ごとに別々の係数がつきます。第3回では、区分に番号を振ると、振り方しだいで係数の向きまで変わってしまうのを見ました。その解決が、この「その区分なら1」の列（ダミー列）です。この部品が作ってくれます。

`oh.transform`の結果は、ほとんどが0なので、0を省いてしまっておく形（疎行列）で返ってきます。`.toarray()`は、それを0も並べたふつうの配列にして、見えるようにするメソッドです。


### 変えてみる：前処理とモデルを、パイプラインに並べる

第3回は、補う値と標準化（standardization）の物差しを、訓練で決める関数`prep`を自分で書きました。ここでは、それを部品の組み合わせで作ります。3つの段階に分けて組み立てます。

段階1は、列の種類ごとの手順です。`Pipeline`は、`('名前', 部品)`の組を、使う順にリストで並べた1本の部品です。

- 数の列：中央値で補う → 標準化
- 区分の列：最頻値（いちばん多い区分）で補う → ワンホット。区分には平均も中央値もないので、最頻値で補います
- `handle_unknown='ignore'`：訓練になかった区分が来ると、ふつうはそこで止まります。`'ignore'`にすると、止まらずに、その区分の列をすべて0にします


In [ ]:
num_steps = Pipeline([('imp', SimpleImputer(strategy='median')),
                      ('sc', StandardScaler())])
cat_steps = Pipeline([
    ('imp', SimpleImputer(strategy='most_frequent')),
    ('oh', OneHotEncoder(handle_unknown='ignore'))])


段階2は、列ごとに別の手順を当てることです。`ColumnTransformer`は、`('名前', 手順, 列のリスト)`の組で、どの列にどの手順を当てるかを決める部品です。


In [ ]:
pre = ColumnTransformer([('num', num_steps, num),
                         ('cat', cat_steps, cat)])


段階3は、前処理とモデルを1本にすることです。もう一度`Pipeline`で、「前処理`pre` → モデル」を1本の部品にまとめます（部品の中に部品が入っています）。`fit`を呼ぶと、前から順に訓練で学びます。


In [ ]:
pipe = Pipeline([('pre', pre),
                 ('model', LogisticRegression(max_iter=5000))])
pipe.fit(X_tr, y_tr)
print(round(roc_auc_score(y_te, pipe.predict_proba(X_te)[:, 1]), 3))


テストのAUCは0.966（前回は0.951）。区分をワンホットにしましたが、差は小さく、61人のテストでは運の範囲でもあります。

大事なのは、`pipe.fit(X_tr, y_tr)`の1行で、補う値も、物差しも、区分の種類も、係数も、すべて訓練だけで学ばれたことです。テストの`X_te`は、学んだ手順を当てられるだけです。第3回で気をつけたデータリーク（data leakage）が、部品の形で防がれています。


---


## 2. fitとpredictの正体


`StandardScaler`や`LogisticRegression`は、どれも`fit`してから`transform`や`predict`を呼びます。この「形」は、どうなっているのでしょう。小さな部品を自分で書いて、中を読んでみます。


### おさらい：def

```python
def 名前(引数):
    return 戻り値
```

`def`の後ろが名前と引数（入力）、`return`の後ろが戻り値（出力）でした（第3回）。


### お手本：標準化の部品を、クラスで書く

- 役割：`fit`で訓練の平均と標準偏差（standard deviation）を覚え、`transform`でその物差しを当てる部品を作る
- 入力：数の列の配列
- 出力：覚えた平均

`class`は、データ（覚えたもの）と、それを使う手順（def）をひとまとめにした型です。中の`def`をメソッドと呼びます。`self`は「この部品自身」で、`self.mean_`と書くと、部品の中に値を覚えておけます。


In [ ]:
class MyScaler:
    def fit(self, X):                  # 訓練で物差しを覚える
        self.mean_ = X.mean(axis=0)
        self.std_ = X.std(axis=0)
        return self
    def transform(self, X):            # 覚えた物差しを当てる
        return (X - self.mean_) / self.std_

A_tr, A_te = X_tr[num].values, X_te[num].values
sc = MyScaler().fit(A_tr)
print(sc.mean_.round(1))


訓練の平均は、年齢54.5歳、血圧131、コレステロール249.8、最大心拍数150、下がり幅1.0。`fit`が物差しを覚え、`return self`で部品そのものを返すので、`MyScaler().fit(A_tr)`のように続けて書けます。


### 変えてみる：scikit-learnのStandardScalerと比べる

同じ配列を、自作の部品と`StandardScaler`で整えて、結果が一致するか確かめます。`np.allclose`は「すべての値がほぼ等しいか」を返します。


In [ ]:
mine = sc.transform(A_te)                      # テストに訓練の物差し
sk = StandardScaler().fit(A_tr).transform(A_te)
print(np.allclose(mine, sk))


`True`です。`StandardScaler`の中身も、「`fit`で覚えて、`transform`で当てる」という、この形をしています。`LogisticRegression`なら、`fit`で覚えるのが係数で、`predict_proba`がそれを当てるメソッドです。パイプラインは、この形の部品を順に呼んでいるだけです。


---


## 3. Cを、交差検証で選ぶ


手順は、前処理からモデルまで1つの部品になったので、次はその中のモデルのつまみを選びます。ロジスティック回帰（logistic regression）にも、第5回の正則化（regularization）と同じ強さのつまみ`C`があります（初期値は1）。`C`が小さいほど、係数を強く縮めます（第5回のalphaとは逆向きです）。これを、交差検証で選びます。


### おさらい：辞書

```python
d = {'キー': 値}
d['キー']          # 値を取り出す
```

名前（キー）と値の組を覚えておく入れ物でした（第3回）。


### お手本：GridSearchCVでCを選ぶ

- 役割：Cの候補を5つ試し、5分割の交差検証のAUCが一番よいものを選ぶ
- 入力：パイプライン`pipe`、候補の辞書、訓練データ
- 出力：選ばれたCと、その交差検証のAUC

候補は辞書で渡します。キーの`'model__C'`は、「パイプラインの`model`という部品の`C`」という意味です（部品の名前と引数の名前を、`_` 2つでつなぎます）。パイプラインごと交差検証するので、補う値や物差しも、分けた訓練の中だけで毎回学び直されます。

選び終えると、`GridSearchCV`は、選んだCで訓練全体（242人）を学び直します。だから`gs`は、そのまま`gs.predict_proba`で当てるのに使えます。


In [ ]:
grid = {'model__C': [0.01, 0.1, 1, 10, 100]}
gs = GridSearchCV(pipe, grid, cv=5, scoring='roc_auc').fit(X_tr, y_tr)
print(gs.best_params_, round(gs.best_score_, 3))
prob = gs.predict_proba(X_te)[:, 1]   # 選んだ C で、テストの確率
print(round(roc_auc_score(y_te, prob), 3))


選ばれたのはC＝0.1で、交差検証のAUCは0.898です。テストのAUCは0.965です。

テストのほうがずっと高く出ました。交差検証は訓練242人を5つに分けて測った平均、テストは61人での1回だけです。テストは人数が少なく、たまたま当てやすい人が集まった可能性があります。0.965を、このモデルのいつもの成績と思わないほうがよいでしょう。


### TODO①：Cの候補ごとの成績を並べる

`gs.cv_results_['mean_test_score']`には、候補の順に交差検証のAUCの平均が入っています。`for`と`zip`で、Cの値とAUCを1行ずつ表示してください。


In [ ]:
# TODO: C の候補ごとに、交差検証の AUC を表示する

# 解答例①：C の候補ごとの成績
for c, s in zip(grid['model__C'], gs.cv_results_['mean_test_score']):
    print(c, round(s, 3))


0.01で0.879、0.1で0.898、1で0.894、10と100で0.890。縮めすぎ（0.01）は少し悪く、0.1から先はほとんど差がありません。このデータでは、Cの選び方で大きくは変わらないことも読み取れます。


---


## 4. しきい値を動かす


ここからは、選んだモデルの確率`prob`を使って、どこで区切るかを考えます。


### おさらい：for

```python
for i in range(3):
    print(i)          # 0, 1, 2 と表示される
```

並びの中身を1つずつ名前に入れながら、字下げした行を繰り返す書き方でした（第2回）。


### お手本：しきい値0.5で、3つの数を数える

- 役割：しきい値0.5で判定し、精密検査に回す人数・見逃し・誤検出を数える
- 入力：確率`prob`、本当の答え`y_te`
- 出力：3つの人数

前回と同じ数え方です。`pred.sum()`は、判定が1（精密検査へ）の人数です。


In [ ]:
t = 0.5
pred = (prob > t).astype(int)
fn = ((pred == 0) & (y_te == 1)).sum()     # 見逃し
fp = ((pred == 1) & (y_te == 0)).sum()     # 誤検出
print(t, pred.sum(), fn, fp)


0.5では、31人を精密検査に回し、見逃し2人、誤検出5人です。


### 変えてみる：3つのしきい値で、くり返す

同じ手順を、`for`で0.3・0.5・0.7に当てます。


In [ ]:
for t in [0.3, 0.5, 0.7]:
    pred = (prob > t).astype(int)
    fn = ((pred == 0) & (y_te == 1)).sum()
    fp = ((pred == 1) & (y_te == 0)).sum()
    print(t, pred.sum(), fn, fp)


しきい値を0.3に下げると、精密検査は38人に増え、見逃しは0人になります。0.7に上げると、誤検出は0人になりますが、見逃しは9人です。見逃しと誤検出は、しきい値の上げ下げで入れかわります。


### TODO②：0.1〜0.9の表にする

しきい値を0.1から0.9まで0.1ずつ動かし、1行ずつ辞書`{'しきい値': t, '精密検査': ..., '見逃し': fn, '誤検出': fp}`をリストに集めて、`pd.DataFrame`で表にしてください（第4回の交差検証の表と同じ作り方です）。


In [ ]:
# TODO: 0.1〜0.9 のしきい値ごとに、辞書を rows に集めて表にする

# 解答例②：0.1〜0.9 の表
rows = []
for t in np.arange(1, 10) / 10:
    pred = (prob > t).astype(int)
    fn = ((pred == 0) & (y_te == 1)).sum()
    fp = ((pred == 1) & (y_te == 0)).sum()
    rows.append({'しきい値': t, '精密検査': pred.sum(),
                 '見逃し': fn, '誤検出': fp})
print(pd.DataFrame(rows))


表にすると、見逃しは0.3までずっと0人、0.4で1人、0.5で2人、0.7で9人と増えていきます。精密検査の人数は、0.1の51人から0.9の8人まで減ります。「見逃し0人」を守れるのは0.3までです。


In [ ]:
# @title 図：しきい値と、見逃し・誤検出
ts = np.arange(1, 10) / 10
fns = [((prob <= t) & (y_te == 1)).sum() for t in ts]
fps = [((prob > t) & (y_te == 0)).sum() for t in ts]
plt.plot(ts, fns, 'o-', lw=2.5, color='#e63946', label='見逃し')
plt.plot(ts, fps, 's-', lw=2.5, color='#0066cc', label='誤検出')
plt.xlabel('しきい値'); plt.ylabel('人数（テスト61人）'); plt.legend()
plt.show()


---


## 5. 誤りの重さから、しきい値を決める


表からは、見逃しと誤検出の入れかわりが見えました。でも、どこで区切るかは、2つの誤りのどちらがどれだけ重いかで決まります。

確率 $p$ の人がいるとします。1回の誤検出の重さを $c_{FP}$、1回の見逃しの重さを $c_{FN}$ とすると、

- 精密検査に回すと、健康だったときに誤検出になる：重さの見込みは $(1-p)\,c_{FP}$
- 帰すと、病気だったときに見逃しになる：重さの見込みは $p\,c_{FN}$

回すほうが軽いのは $p\,c_{FN} > (1-p)\,c_{FP}$ のとき。これを $p$ について解くと、

$$
p > t^* = \frac{c_{FP}}{c_{FP} + c_{FN}}
$$

確率がこの $t^*$ を超えたら、精密検査に回すのが得です。


### お手本：2つの誤りが同じ重さなら

- 役割：誤検出と見逃しを同じ重さ（1と1）として、$t^*$ を計算する
- 入力：2つの重さ
- 出力：しきい値 $t^*$


In [ ]:
c_fp, c_fn = 1, 1            # 誤検出と見逃しの重さ
print(c_fp / (c_fp + c_fn))


0.5です。前回からずっと使ってきた0.5は、「2つの誤りを同じ重さと見る」という選択だったのです。


### TODO③：見逃しが4倍重いなら

健診では、見逃しのほうがずっと重い誤りです。見逃しの重さを誤検出の4倍（`c_fn = 4`）として、$t^*$ を計算してください。そして、TODO②の表で、そのしきい値の見逃しと精密検査の人数を読んでください。


In [ ]:
# TODO: c_fn = 4 のときの t* を計算する

# 解答例③：見逃しが4倍重いとき
c_fp, c_fn = 1, 4
print(c_fp / (c_fp + c_fn))


$t^*$ ＝ 1 ÷ (1 ＋ 4) ＝ 0.2。表では、0.2のとき見逃し0人、精密検査42人です。

0.3でも見逃しは0人で、精密検査は38人と少なくすみます。でも、0.2から0.3で減った4人は、たまたま全員健康だっただけです。61人のテストでは、こうした偶然で表が変わります。

そこで、訓練でも確かめます。訓練の一人ひとりに、その人を外して学んだモデルで確率を出します（交差検証と同じ分け方）。誤検出＋見逃し×4で誤りの重さを合計すると、0.2なら83、0.3なら91です。訓練で見ると、0.2のほうが軽くなります。

$t^*$ の計算は、確率が言葉どおりであることを前提にしています。「確率0.2」と出た人を集めると、本当にその2割が病気だったということです。モデルの確率がこうなっているとき、確率の較正がとれていると言います（前回の発展で見た図です）。較正がずれていれば、$t^*$ もずれます。

だから、$t^*$ は答えそのものではなく、誤りの重さという考えから決める出発点です。そこから、表で人数を確かめます。ここでは、見逃しを避けるほうを選んで0.2を採ります。


---


## 6. 新しい受診者を判定する


`C`を選んだ`gs`と、しきい値0.2がそろいました。最初の問いに戻ります。新しい受診者が来ました。検査のうち、血管の検査（`ca`）と負荷試験（`thal`）は、まだ結果が出ていません。


### お手本：新しい1人を、同じ手順で判定する

- 役割：新しい受診者1人の検査値を表にし、パイプラインで確率を出して、しきい値0.2で判定する
- 入力：検査値の辞書
- 出力：心臓病である確率と、判定

結果の出ていない検査は`np.nan`（欠けた値）にします。パイプラインが、訓練で覚えた値で補い、訓練の物差しで整えてから、確率を出します。

判定の文字は、1行の`if〜else`で選びます。`A if 条件 else B`は、条件が成り立てばA、成り立たなければBになる書き方です。

```python
x = 5
print('大' if x > 3 else '小')   # 大 と表示される
```


In [ ]:
new = pd.DataFrame([{'age': 58, 'trestbps': 140, 'chol': 250,
                     'thalach': 130, 'oldpeak': 1.5, 'sex': 1,
                     'cp': 4, 'fbs': 0, 'restecg': 2, 'exang': 1,
                     'slope': 2, 'ca': np.nan, 'thal': np.nan}])
p = gs.predict_proba(new)[0, 1]
print(round(p, 2), '精密検査へ' if p > 0.2 else '経過観察')


確率は0.72。しきい値0.2を大きく超えるので、精密検査に回します。前処理を自分で書き直さなくても、`gs.predict_proba(new)`の1行で、訓練と同じ手順が当てられました。


### TODO④：もう1人を判定する

2人目の受診者は、お手本の人と比べて、胸の痛みの型が`cp=3`、最大心拍数が`thalach=165`、運動で胸が痛まない（`exang=0`）、心電図の下がり幅が`oldpeak=0.2`です。ほかは同じです。確率を出し、しきい値0.2と0.5で、それぞれ判定してください。


In [ ]:
# TODO: 2人目の確率を出し、0.2 と 0.5 で判定する

# 解答例④：2人目の判定
new2 = new.assign(cp=3, thalach=165, exang=0, oldpeak=0.2)
p2 = gs.predict_proba(new2)[0, 1]
print(round(p2, 2))
for t in [0.2, 0.5]:
    print(t, '精密検査へ' if p2 > t else '経過観察')


確率は0.22。しきい値0.2なら精密検査に回り、0.5なら帰されます。しきい値を決めることは、こうした境目の人をどちらに送るかを決めることです。

答えは「前処理とモデルを1本のパイプラインにまとめれば、新しい受診者にも訓練と同じ手順を当てられる。見逃しを誤検出の4倍重いと見るなら、確率0.2を超えた人を精密検査に回す。テスト61人では、見逃し0人、精密検査42人になる」。

まだ言えないことも、3つ残っています。

- ほかのモデルなら：ロジスティック回帰のほかの部品に差し替えたら、どうなるか（課題）
- 重さの数：「見逃しは4倍」は、医師と患者が話し合って決める値で、データからは出てこない
- 確率は信じてよいか：$t^*$ は確率が言葉どおり（較正がとれている）ときの区切り（前回の発展）


---


## 目標に答えられたか


- 今日の目標は「新しい受診者を同じ手順で判定し、どの確率から精密検査に回すかを決める」ことでした。6節の2人は、それぞれどう判定されましたか？
- 1節で、`cp`を番号のまま入れると何が困りましたか？
- 2節の`MyScaler`で、`fit`と`transform`はそれぞれ何をしていましたか？
- 5節で、しきい値0.5は、2つの誤りの重さをどう見ることでしたか？


---


## 今日の要点


- 番号で書かれた区分は、ワンホットで「その区分なら1」の列に分ける
- `ColumnTransformer`で列ごとに前処理を分け、`Pipeline`で前処理とモデルを1本にまとめる。`fit`は訓練だけで学ぶ
- GridSearchCVは、パイプラインごと交差検証してCを選ぶ。候補は辞書で渡す
- しきい値を下げると見逃しが減り、誤検出が増える。表にして入れかわりを見る
- 誤りの重さから、$t^* = c_{FP} / (c_{FP} + c_{FN})$。0.5は「同じ重さ」という選択
- Python：`class`は覚えたもの（`self.〜`）と手順（メソッド）をまとめた型。scikit-learnの部品は、`fit`で覚えて`transform`・`predict`で当てる形


---


## 次回へ


今日で、教師あり学習の道具（前処理・モデル・交差検証・評価・しきい値）がそろいました。

次回は総合演習です。新しいデータで、データの理解から報告までを一周します。そして、自分のコードを、関数に分けて1本にまとめます。

次回の問いは「測定値からペンギンの3種を見分けられるか？ どの測定が、見分けに役立つか？」です。


---


## 課題（提出）


提出するもの：応用②の答えと、応用③の文章。提出フォームに入力してください。期限はありません。応用①のコードは提出しませんが、②の答えを出すために必要です。


### 応用①（変形）

パイプラインの`model`の部品を、ロジスティック回帰からランダムフォレスト（`RandomForestClassifier(random_state=42)`）に差し替えてください。候補の辞書を`{'model__max_depth': [3, 5, None], 'model__n_estimators': [100, 300]}`としてGridSearchCVで選び、選ばれた組み合わせと、テストのAUCを表示してください。

前処理の部品`pre`は、そのまま使えます。


In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = Pipeline([('pre', pre),
               ('model', RandomForestClassifier(random_state=42))])
grid_rf = {'model__max_depth': [3, 5, None],
           'model__n_estimators': [100, 300]}
gs_rf = GridSearchCV(rf, grid_rf, cv=5, scoring='roc_auc').fit(X_tr, y_tr)
prob_rf = gs_rf.predict_proba(X_te)[:, 1]
print(gs_rf.best_params_)
print(round(roc_auc_score(y_te, prob_rf), 3))


<details><summary>詰まったら</summary>

`from sklearn.ensemble import RandomForestClassifier`で呼び出し、
`Pipeline([('pre', pre), ('model', RandomForestClassifier(random_state=42))])`を作ります。
あとは3節と同じく`GridSearchCV(…, cv=5, scoring='roc_auc').fit(X_tr, y_tr)`です。

</details>


### 応用②（判断）

応用①のランダムフォレストのテストのAUCは、3節のロジスティック回帰（0.965）より上ですか、下ですか。「上」か「下」で答えてください。


In [ ]:
print('答え:', '上' if roc_auc_score(y_te, prob_rf) > 0.965 else '下')


<details><summary>詰まったら</summary>

`roc_auc_score(y_te, 確率)`を0.965と比べます。

</details>


### 応用③（解釈）

ロジスティック回帰とランダムフォレストのどちらを健診に使うかを、健診センターの運営担当者に向けて3行程度で書いてください。テストのAUCと、「係数で理由を説明できるか」に触れてください。


（模範例）

テストのAUCは、ロジスティック回帰が0.965、ランダムフォレストが0.948でした。訓練の交差検証では0.898と0.903で順位が逆になり、差はどちらにもわずかです。

ロジスティック回帰は、どの検査値が確率を押し上げたかを係数で説明できるので、受診者に判定の理由を伝えやすい方法です。

成績に大きな差がないので、健診ではロジスティック回帰を使い、理由を説明できる判定にするほうがよいと考えます。


---


## 発展（任意）


### LightGBMを、同じパイプラインに載せる

LightGBMは、小さな決定木を1本ずつ足して、前の木の間違いを次の木で直していく方法（勾配ブースティング）です。表のデータでは、とても強い方法として知られています。

パイプラインの良いところは、`model`の部品を差し替えるだけで、前処理はそのまま使えることです。


In [ ]:
!pip install -q lightgbm
from lightgbm import LGBMClassifier

model = LGBMClassifier(n_estimators=200, learning_rate=0.05,
                       random_state=42, verbose=-1)
lgb = Pipeline([('pre', pre), ('model', model)])
lgb.fit(X_tr, y_tr)
print(round(roc_auc_score(y_te, lgb.predict_proba(X_te)[:, 1]), 3))


読み方　テストのAUCは、ロジスティック回帰とほとんど変わりません。303人・13本の検査という小さな表では、強い方法でも差が出にくいのです。

強い方法ほど、「なぜその確率なのか」を係数で言うのは難しくなります。差がわずかなら、説明しやすい方法を選ぶという判断も十分にありえます。

試すなら、`learning_rate`を0.01や0.2に変えて、AUCがどう動くか見てみましょう。
